# Wan 2.2 — your free AI video studio (Google Colab)

Type a sentence, get a 5-second vertical video, made by **Wan 2.2 (5B)** —
the model whose detective clip you liked — running on Google's free GPU.
Nothing installs on your PC. Nothing costs money.

## Before you press anything — ONE setting
Top menu: **Runtime → Change runtime type → Hardware accelerator: GPU (T4)** → Save.
Without this, the first cell stops and tells you.

## How to use it
1. Run the cells **in order, top to bottom** (hover a cell, click its ▶ play button).
2. Steps 1–3 are setup. They take about **10 minutes** the first time.
3. The **last cell is the fun one**: change the `PROMPT`, run it, get a video.
   After setup, making another video = run only that last cell again.

## Honest warnings
- The free GPU is **slow**: one 5-second clip can take **15–40 minutes**. Free is free.
- If you close the tab or the session sleeps, Colab **deletes everything** —
  download your video as soon as it appears (last cell shows it with a download option).
- Setup runs again each new session. That's the deal with free.

In [ ]:
# GPU check — this must say a GPU name (usually Tesla T4)
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU FOUND. Go to Runtime -> Change runtime type -> GPU (T4), Save, then run this cell again.")
print("GPU OK:", torch.cuda.get_device_name(0))

## Step 1 — Install the studio (ComfyUI)
Takes about 3–5 minutes. Lots of text will scroll — that's normal.

In [ ]:
%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
print("Install done.")

## Step 2 — Download the Wan 2.2 brain (about 18 GB total)
Three files: the video model, the text reader, and the image decoder.
On Colab's fast internet this takes roughly 3–8 minutes.

In [ ]:
%cd /content/ComfyUI
!mkdir -p models/diffusion_models models/text_encoders models/vae
BASE = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
!wget -c -q --show-progress "{BASE}/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors" -P models/diffusion_models/
!wget -c -q --show-progress "{BASE}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors" -P models/text_encoders/
!wget -c -q --show-progress "{BASE}/vae/wan2.2_vae.safetensors" -P models/vae/
print("Models downloaded.")

## Step 3 — Start the engine
Wait until it prints **Engine is ready**.

In [ ]:
import subprocess, time, urllib.request
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for i in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print("Engine is ready.")
        break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("Engine did not start in 3 minutes — run this cell once more.")

## Step 4 — MAKE YOUR VIDEO
Change the **PROMPT** to anything you imagine. Run this cell, then wait —
the free GPU needs roughly **15–40 minutes** for 5 seconds of video.
The finished video appears at the bottom of this cell. **Download it right away**
(the player has a download button) — Colab deletes everything when the session ends.

Want a different result with the same prompt? Change `SEED` to any other number.
Too slow or "out of memory"? In the cell below, `length: 121` counts the frames —
lower it to `81` (about 3.4 seconds) and run again.

In [ ]:
# ---- WRITE YOUR VIDEO IDEA HERE ----
PROMPT = ("3D animation, cute Pixar style, a tiny orange cat wearing a small "
          "purple wizard hat sits on a wooden stool in a cozy magic shop, "
          "it waves one paw at the camera, blinks and smiles, warm golden "
          "light, colorful potions on shelves behind it, smooth motion, "
          "high quality")
SEED = 42
# -------------------------------------

import json, time, urllib.request, shutil, os
from IPython.display import Video, display

workflow = json.loads(r'''{"37": {"class_type": "UNETLoader", "inputs": {"unet_name": "wan2.2_ti2v_5B_fp16.safetensors", "weight_dtype": "default"}}, "38": {"class_type": "CLIPLoader", "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}}, "39": {"class_type": "VAELoader", "inputs": {"vae_name": "wan2.2_vae.safetensors"}}, "48": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["37", 0], "shift": 8.0}}, "6": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["38", 0], "text": "placeholder — set by the cell"}}, "7": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["38", 0], "text": "\u8272\u8c03\u8273\u4e3d\uff0c\u8fc7\u66dd\uff0c\u9759\u6001\uff0c\u7ec6\u8282\u6a21\u7cca\u4e0d\u6e05\uff0c\u5b57\u5e55\uff0c\u98ce\u683c\uff0c\u4f5c\u54c1\uff0c\u753b\u4f5c\uff0c\u753b\u9762\uff0c\u9759\u6b62\uff0c\u6574\u4f53\u53d1\u7070\uff0c\u6700\u5dee\u8d28\u91cf\uff0c\u4f4e\u8d28\u91cf\uff0cJPEG\u538b\u7f29\u6b8b\u7559\uff0c\u4e11\u964b\u7684\uff0c\u6b8b\u7f3a\u7684\uff0c\u591a\u4f59\u7684\u624b\u6307\uff0c\u753b\u5f97\u4e0d\u597d\u7684\u624b\u90e8\uff0c\u753b\u5f97\u4e0d\u597d\u7684\u8138\u90e8\uff0c\u7578\u5f62\u7684\uff0c\u6bc1\u5bb9\u7684\uff0c\u5f62\u6001\u7578\u5f62\u7684\u80a2\u4f53\uff0c\u624b\u6307\u878d\u5408\uff0c\u9759\u6b62\u4e0d\u52a8\u7684\u753b\u9762\uff0c\u6742\u4e71\u7684\u80cc\u666f\uff0c\u4e09\u6761\u817f\uff0c\u80cc\u666f\u4eba\u5f88\u591a\uff0c\u5012\u7740\u8d70"}}, "55": {"class_type": "Wan22ImageToVideoLatent", "inputs": {"vae": ["39", 0], "width": 704, "height": 1280, "length": 121, "batch_size": 1}}, "3": {"class_type": "KSampler", "inputs": {"model": ["48", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["55", 0], "seed": 0, "steps": 20, "cfg": 5.0, "sampler_name": "uni_pc", "scheduler": "simple", "denoise": 1.0}}, "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["39", 0]}}, "57": {"class_type": "CreateVideo", "inputs": {"images": ["8", 0], "fps": 24.0}}, "58": {"class_type": "SaveVideo", "inputs": {"video": ["57", 0], "filename_prefix": "video/ComfyUI", "format": "auto", "codec": "auto"}}}''')
workflow["6"]["inputs"]["text"] = PROMPT
workflow["3"]["inputs"]["seed"] = SEED

req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=json.dumps({"prompt": workflow}).encode(),
    headers={"Content-Type": "application/json"})
prompt_id = json.loads(urllib.request.urlopen(req).read())["prompt_id"]
print("Making your video... the free GPU is slow, give it 15-40 minutes.")

while True:
    hist = json.loads(urllib.request.urlopen(
        f"http://127.0.0.1:8188/history/{prompt_id}").read())
    if prompt_id in hist:
        break
    time.sleep(15)

found = None
for out in hist[prompt_id]["outputs"].values():
    for items in out.values():
        if isinstance(items, list):
            for it in items:
                if isinstance(it, dict) and it.get("filename", "").endswith((".mp4", ".webm")):
                    found = os.path.join("/content/ComfyUI/output",
                                         it.get("subfolder", ""), it["filename"])
if not found:
    raise SystemExit("No video came out — scroll up for an error, then run this cell again.")
shutil.copy(found, "/content/my_video.mp4")
print("DONE — your video:")
display(Video("/content/my_video.mp4", embed=True, width=360))

## Optional — keep your videos in Google Drive
Run the cell below once, approve the Google sign-in, and every finished video
also gets copied to a **WanVideos** folder in your Drive, so Colab deleting the
session can't take your clips with it.

In [ ]:
# OPTIONAL: save videos to your Google Drive
from google.colab import drive
import shutil, os
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/WanVideos", exist_ok=True)
if os.path.exists("/content/my_video.mp4"):
    shutil.copy("/content/my_video.mp4", "/content/drive/MyDrive/WanVideos/")
    print("Saved to Drive > WanVideos. Re-run this cell after each new video.")
else:
    print("Drive ready. Make a video first, then re-run this cell to save it.")